In [2]:
import kagglehub
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

path1 = kagglehub.dataset_download("arsalan9702/tickharm-pre-processed-audio")
path2 = kagglehub.dataset_download("aryansraut/preprocessed-ucf-crime-dataset-visual")

VISUAL_ROOT = "/kaggle/input/datasets/aryansraut/preprocessed-ucf-crime-dataset-visual/TikHarm_frames_16"
AUDIO_ROOT  = "/kaggle/input/datasets/arsalan9702/tickharm-pre-processed-audio/TikHarm_audio"

VISUAL_CKPT = "/kaggle/input/models/arsalan9702/tikharm-models/pytorch/default/1/best_swin3d_tikharm.pt"
AUDIO_CKPT  = "/kaggle/input/models/arsalan9702/tikharm-models/pytorch/default/1/best_audio_cnn14.pth"

In [3]:
!pip install torchlibrosa -q

In [4]:
from torchlibrosa.stft import Spectrogram, LogmelFilterBank

class CNN14(nn.Module):
    def __init__(self):
        super().__init__()

        self.spectrogram_extractor = Spectrogram(n_fft=1024, hop_length=320, win_length=1024)
        self.logmel_extractor = LogmelFilterBank(sr=16000, n_fft=1024, n_mels=64)

        self.bn0 = nn.BatchNorm2d(64)

        def block(i,o):
            return nn.Sequential(
                nn.Conv2d(i,o,3,1,1),
                nn.BatchNorm2d(o),
                nn.ReLU(),
                nn.MaxPool2d(2)
            )

        self.conv_block1 = block(1,64)
        self.conv_block2 = block(64,128)
        self.conv_block3 = block(128,256)
        self.conv_block4 = block(256,512)

        self.fc1 = nn.Linear(512,512)
        self.fc_out = nn.Linear(512,4)

    def forward(self,x):
        x = self.spectrogram_extractor(x)
        x = self.logmel_extractor(x)
        x = x.transpose(1,3)
        x = self.bn0(x)
        x = x.transpose(1,3)

        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.conv_block4(x)

        x = torch.mean(x,3)
        x = torch.mean(x,2)

        feat = F.relu(self.fc1(x))
        return self.fc_out(feat)

In [5]:
class FusionDataset(Dataset):
    def __init__(self, split):
        self.samples = []
        self.max_audio = 16000*10

        self.transform = T.Compose([
            T.Resize((224,224)),
            T.ToTensor(),
            T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        ])

        classes = ["Adult Content","Harmful Content","Safe","Suicide"]
        self.map = {c:i for i,c in enumerate(classes)}

        for cls in classes:
            v = os.path.join(VISUAL_ROOT, split, cls)
            a = os.path.join(AUDIO_ROOT, split, cls)

            for vid in os.listdir(v):
                vp = os.path.join(v,vid)
                ap = os.path.join(a,vid+".wav")
                if os.path.exists(ap):
                    self.samples.append((vp,ap,self.map[cls]))

    def __len__(self): return len(self.samples)

    def __getitem__(self,i):
        vp,ap,y = self.samples[i]

        frames = sorted(os.listdir(vp))
        idxs = torch.linspace(0,len(frames)-1,16).long()

        imgs=[]
        for j in idxs:
            img = Image.open(os.path.join(vp,frames[j])).convert("RGB")
            imgs.append(self.transform(img))

        video = torch.stack(imgs).permute(1,0,2,3)

        wav,sr = torchaudio.load(ap)
        if sr!=16000:
            wav = torchaudio.functional.resample(wav,sr,16000)

        wav = wav.mean(0)
        if wav.shape[0]<self.max_audio:
            wav = F.pad(wav,(0,self.max_audio-wav.shape[0]))
        else:
            wav = wav[:self.max_audio]

        return video,wav,y

In [6]:
train_loader = DataLoader(FusionDataset("train"), batch_size=8, shuffle=True, num_workers=0)
val_loader   = DataLoader(FusionDataset("val"), batch_size=8)
test_loader  = DataLoader(FusionDataset("test"), batch_size=8)

In [8]:
visual_model = torch.hub.load("pytorch/vision:v0.15.2","swin3d_t",pretrained=False)
visual_model.head = nn.Linear(visual_model.head.in_features,4)
visual_model.load_state_dict(torch.load(VISUAL_CKPT)["model_state_dict"])
visual_model.to(device).eval()

audio_model = CNN14()
audio_model.load_state_dict(torch.load(AUDIO_CKPT))
audio_model.to(device).eval()

Using cache found in /root/.cache/torch/hub/pytorch_vision_v0.15.2
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


CNN14(
  (spectrogram_extractor): Spectrogram(
    (stft): STFT(
      (conv_real): Conv1d(1, 513, kernel_size=(1024,), stride=(320,), bias=False)
      (conv_imag): Conv1d(1, 513, kernel_size=(1024,), stride=(320,), bias=False)
    )
  )
  (logmel_extractor): LogmelFilterBank()
  (bn0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv_block1): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block2): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block3): Sequential(
    (0)

In [17]:
def vfeat(x):
    model = visual_model.module if isinstance(visual_model, nn.DataParallel) else visual_model

    x = model.patch_embed(x)
    x = model.pos_drop(x)

    for layer in model.features:
        x = layer(x)

    x = model.norm(x)

    # x shape: (B, T, H, W, C)
    x = x.permute(0, 4, 1, 2, 3)   # → (B, C, T, H, W)

    x = torch.mean(x, dim=[2, 3, 4])  # global avg pool

    return x

def afeat(x):
    x = audio_model.spectrogram_extractor(x)
    x = audio_model.logmel_extractor(x)
    x = x.transpose(1,3)
    x = audio_model.bn0(x)
    x = x.transpose(1,3)

    x = audio_model.conv_block1(x)
    x = audio_model.conv_block2(x)
    x = audio_model.conv_block3(x)
    x = audio_model.conv_block4(x)

    x = torch.mean(x,3)
    x = torch.mean(x,2)

    return F.relu(audio_model.fc1(x))

In [18]:
class Fusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(768+512,256),
            nn.ReLU(),
            nn.Linear(256,4)
        )

    def forward(self,v,a):
        return self.net(torch.cat([v,a],1))

fusion_model = Fusion().to(device)
opt = torch.optim.Adam(fusion_model.parameters(),1e-3)
loss_fn = nn.CrossEntropyLoss()

In [19]:
def train():
    fusion_model.train()
    for v,a,y in tqdm(train_loader):
        v,a,y = v.to(device),a.to(device),y.to(device)

        with torch.no_grad():
            vf = vfeat(v)
            af = afeat(a)

        out = fusion_model(vf,af)
        loss = loss_fn(out,y)

        opt.zero_grad()
        loss.backward()
        opt.step()

def evaluate(loader):
    fusion_model.eval()
    c,t=0,0
    with torch.no_grad():
        for v,a,y in loader:
            v,a,y = v.to(device),a.to(device),y.to(device)
            vf = vfeat(v)
            af = afeat(a)
            p = fusion_model(vf,af).argmax(1)
            c+=(p==y).sum().item()
            t+=y.size(0)
    return c/t

In [20]:
for epoch in range(5):
    train()

    val_acc = evaluate(val_loader)
    test_acc = evaluate(test_loader)

    print(f"Epoch {epoch+1}")
    print("Val:",val_acc)
    print("Test:",test_acc)
    print("-"*30)

100%|██████████| 346/346 [08:33<00:00,  1.48s/it]


Epoch 1
Val: 0.8661616161616161
Test: 0.8556962025316456
------------------------------


100%|██████████| 346/346 [06:10<00:00,  1.07s/it]


Epoch 2
Val: 0.8459595959595959
Test: 0.8481012658227848
------------------------------


100%|██████████| 346/346 [06:12<00:00,  1.08s/it]


Epoch 3
Val: 0.8661616161616161
Test: 0.8556962025316456
------------------------------


100%|██████████| 346/346 [06:17<00:00,  1.09s/it]


Epoch 4
Val: 0.8636363636363636
Test: 0.8620253164556962
------------------------------


100%|██████████| 346/346 [06:32<00:00,  1.14s/it]


Epoch 5
Val: 0.8636363636363636
Test: 0.8670886075949367
------------------------------


In [21]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np

def full_report(loader, model_type="fusion_mlp"):
    y_true = []
    y_pred = []

    fusion_model.eval()
    visual_model.eval()
    audio_model.eval()

    with torch.no_grad():
        for video, audio, labels in loader:
            video = video.to(device)
            audio = audio.to(device)
            labels = labels.to(device)

            # ---- FEATURES ----
            if model_type in ["feature", "transformer"]:
                vf = vfeat(video)
                af = afeat(audio)
                logits = fusion_model(vf, af)

            elif model_type == "mlp":
                audio = audio.squeeze(1)
                v_logits = visual_model(video)
                a_logits = audio_model(audio)
                logits = fusion_model(v_logits, a_logits)

            elif model_type == "alpha":
                audio = audio.squeeze(1)
                v_logits = visual_model(video)
                a_logits = audio_model(audio)
                logits = 0.8 * v_logits + 0.2 * a_logits

            preds = torch.argmax(logits, dim=1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    classes = ["Adult Content", "Harmful Content", "Safe", "Suicide"]

    print("\n=== Accuracy ===")
    print(accuracy_score(y_true, y_pred))

    print("\n=== Classification Report ===")
    print(classification_report(y_true, y_pred, target_names=classes))

    print("\n=== Confusion Matrix ===")
    print(confusion_matrix(y_true, y_pred))

In [22]:
full_report(test_loader, model_type="feature")      # feature fusion
# full_report(test_loader, model_type="transformer") # transformer
# full_report(test_loader, model_type="mlp")         # old MLP
# full_report(test_loader, model_type="alpha")       # alpha baseline


=== Accuracy ===
0.8670886075949367

=== Classification Report ===
                 precision    recall  f1-score   support

  Adult Content       0.85      0.89      0.87       195
Harmful Content       0.88      0.78      0.82       198
           Safe       0.90      0.88      0.89       200
        Suicide       0.84      0.93      0.88       197

       accuracy                           0.87       790
      macro avg       0.87      0.87      0.87       790
   weighted avg       0.87      0.87      0.87       790


=== Confusion Matrix ===
[[173   9   3  10]
 [ 18 154  10  16]
 [  7  10 175   8]
 [  5   3   6 183]]
